# TikTok Video Claims Classification

## End-to-End Machine Learning Project

Business problem: help TikTok moderators prioritize user reports by predicting whether a video contains a claim or an opinion. A claim prediction is not a fact-check or enforcement decision.

The supplied dataset contains one row per video. The claim_status column is the target. This is supervised binary classification because past labelled examples are used to predict one of two categories for unseen videos.

Tanglish: model claim or opinion nu classify pannum. Claim true-ah false-ah nu decide pannaadhu.


## 1. Imports

In [3]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

## 2. Data Loading and Inspection
Every inspection action below has its own code cell.

In [5]:
DATA_PATH = Path('data/tiktok_dataset.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('project/data/tiktok_dataset.csv')
df = pd.read_csv(DATA_PATH)
print(f'Loaded {DATA_PATH}: {df.shape[0]:,} rows x {df.shape[1]} columns')

Loaded project/data/tiktok_dataset.csv: 19,382 rows x 12 columns


### First five rows
This is calculated from the actual dataset.

In [7]:
df.head()

### Last five rows
This is calculated from the actual dataset.

In [9]:
df.tail()

### Dataset dimensions
This is calculated from the actual dataset.

In [11]:
print(df.shape)

(19382, 12)


### Column names
This is calculated from the actual dataset.

In [13]:
print(df.columns.tolist())

['#', 'claim_status', 'video_id', 'video_duration_sec', 'video_transcription_text', 'verified_status', 'author_ban_status', 'video_view_count', 'video_like_count', 'video_share_count', 'video_download_count', 'video_comment_count']


### Data types
This is calculated from the actual dataset.

In [15]:
df.dtypes

### Dataset information
This is calculated from the actual dataset.

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19382 entries, 0 to 19381
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   #                         19382 non-null  int64  
 1   claim_status              19084 non-null  object 
 2   video_id                  19382 non-null  int64  
 3   video_duration_sec        19382 non-null  int64  
 4   video_transcription_text  19084 non-null  object 
 5   verified_status           19382 non-null  object 
 6   author_ban_status         19382 non-null  object 
 7   video_view_count          19084 non-null  float64
 8   video_like_count          19084 non-null  float64
 9   video_share_count         19084 non-null  float64
 10  video_download_count      19084 non-null  float64
 11  video_comment_count       19084 non-null  float64
dtypes: float64(5), int64(3), object(4)
memory usage: 1.8+ MB


### Numerical descriptive statistics
This is calculated from the actual dataset.

In [19]:
df.describe().T

### Categorical descriptive statistics
This is calculated from the actual dataset.

In [21]:
df.describe(include='object').T

### Missing values
This is calculated from the actual dataset.

In [23]:
df.isna().sum().sort_values(ascending=False)

### Missing-value percentages
This is calculated from the actual dataset.

In [25]:
(df.isna().mean()*100).round(2).sort_values(ascending=False)

### Duplicate records
This is calculated from the actual dataset.

In [27]:
print('Duplicates:', df.duplicated().sum())

Duplicates: 0


### Target values
This is calculated from the actual dataset.

In [29]:
print(df['claim_status'].dropna().unique())

['claim' 'opinion']


### Target distribution
This is calculated from the actual dataset.

In [31]:
df['claim_status'].value_counts(dropna=False)

### Target percentages
This is calculated from the actual dataset.

In [33]:
(df['claim_status'].value_counts(normalize=True, dropna=False)*100).round(2)

### Potential outliers
IQR flags are reviewed, not removed automatically. Viral engagement can be genuine.

In [35]:
engagement_cols = ['video_view_count','video_like_count','video_share_count','video_download_count','video_comment_count']
rows = []
for col in engagement_cols:
    q1,q3 = df[col].quantile([.25,.75])
    upper = q3 + 1.5*(q3-q1)
    rows.append([col,upper,int((df[col]>upper).sum())])
pd.DataFrame(rows,columns=['feature','IQR upper bound','flagged rows'])

## 3. Cleaning
Rows with a missing target cannot be used in supervised learning. The row number and video identifier are excluded from modelling.

In [37]:
model_df = df.dropna(subset=['claim_status']).copy()
print('Retained:',len(model_df),'Removed:',len(df)-len(model_df))

Retained: 19084 Removed: 298


### Missing-value check after target cleaning

In [39]:
model_df.isna().sum().sort_values(ascending=False)

### Standardize categorical values

In [41]:
for col in ['claim_status','verified_status','author_ban_status']:
    model_df[col] = model_df[col].astype(str).str.strip().str.lower()
print({col:model_df[col].unique().tolist() for col in ['claim_status','verified_status','author_ban_status']})

{'claim_status': ['claim', 'opinion'], 'verified_status': ['not verified', 'verified'], 'author_ban_status': ['under review', 'active', 'banned']}


### Check invalid negative values

In [43]:
print({col:int((model_df[col]<0).sum()) for col in ['video_duration_sec']+engagement_cols})

{'video_duration_sec': 0, 'video_view_count': 0, 'video_like_count': 0, 'video_share_count': 0, 'video_download_count': 0, 'video_comment_count': 0}


## 4. Exploratory Data Analysis
EDA-na model munnaadi data patterns understand pannradhu.

### Target class balance
Interpret as association, not causation.

In [46]:
model_df['claim_status'].value_counts(normalize=True).mul(100).round(2)

### Numeric distribution
Interpret as association, not causation.

In [48]:
model_df[['video_duration_sec']+engagement_cols].agg(['mean','median','std','min','max']).T

### Category frequencies
Interpret as association, not causation.

In [50]:
for col in ['verified_status','author_ban_status']:
 print('\n',col); display(model_df[col].value_counts())


 verified_status

 author_ban_status


### Engagement by class
Interpret as association, not causation.

In [52]:
model_df.groupby('claim_status')[engagement_cols].agg(['mean','median']).round(2)

### Duration by class
Interpret as association, not causation.

In [54]:
model_df.groupby('claim_status')['video_duration_sec'].agg(['mean','median','min','max']).round(2)

### Views by class
Interpret as association, not causation.

In [56]:
model_df.groupby('claim_status')['video_view_count'].agg(['mean','median']).round(2)

### Likes by class
Interpret as association, not causation.

In [58]:
model_df.groupby('claim_status')['video_like_count'].agg(['mean','median']).round(2)

### Comments by class
Interpret as association, not causation.

In [60]:
model_df.groupby('claim_status')['video_comment_count'].agg(['mean','median']).round(2)

### Shares by class
Interpret as association, not causation.

In [62]:
model_df.groupby('claim_status')['video_share_count'].agg(['mean','median']).round(2)

### Correlation analysis
Interpret as association, not causation.

In [64]:
model_df[['video_duration_sec']+engagement_cols].corr().round(3)

### Numeric association with target
Interpret as association, not causation.

In [66]:
model_df[['video_duration_sec']+engagement_cols].corrwith((model_df['claim_status']=='claim').astype(int)).sort_values(ascending=False).round(3)

### Claim status by verification
Interpret as association, not causation.

In [68]:
pd.crosstab(model_df['verified_status'],model_df['claim_status'],normalize='columns').mul(100).round(2)

### Claim status by ban status
Interpret as association, not causation.

In [70]:
pd.crosstab(model_df['author_ban_status'],model_df['claim_status'],normalize='columns').mul(100).round(2)

## 5. Visualizations

### Target distribution

In [73]:
ax=sns.countplot(data=model_df,x='claim_status',order=['claim','opinion'])
ax.set(title='Target Distribution',xlabel='Claim status',ylabel='Videos')
for x in ax.containers: ax.bar_label(x)
plt.show()

### Numerical distributions

In [75]:
fig,axes=plt.subplots(2,3,figsize=(15,8))
for ax,col in zip(axes.flat,['video_duration_sec']+engagement_cols):
    sns.histplot(model_df[col],bins=35,ax=ax)
    if col!='video_duration_sec': ax.set_xscale('symlog')
    ax.set_title(col.replace('_',' ').title())
plt.tight_layout();plt.show()

### Outlier boxplots

In [77]:
fig,axes=plt.subplots(1,2,figsize=(13,4))
sns.boxplot(data=model_df,x='claim_status',y='video_view_count',ax=axes[0]);axes[0].set_yscale('symlog')
sns.boxplot(data=model_df,x='claim_status',y='video_like_count',ax=axes[1]);axes[1].set_yscale('symlog')
plt.tight_layout();plt.show()

### Engagement comparison

In [79]:
z=model_df.groupby('claim_status')[engagement_cols].median().T.reset_index().melt(id_vars='index',var_name='claim_status',value_name='median')
sns.barplot(data=z,x='index',y='median',hue='claim_status')
plt.yscale('symlog');plt.xticks(rotation=30,ha='right');plt.tight_layout();plt.show()

### Correlation heatmap

In [81]:
sns.heatmap(model_df[['video_duration_sec']+engagement_cols].corr(),annot=True,fmt='.2f',cmap='vlag',center=0)
plt.title('Numerical Correlation');plt.tight_layout();plt.show()

### Views and likes

In [83]:
sns.scatterplot(data=model_df,x='video_view_count',y='video_like_count',hue='claim_status',alpha=.25,s=18)
plt.xscale('symlog');plt.yscale('symlog');plt.title('Views versus Likes');plt.show()

### Target distribution by account status

In [85]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
sns.countplot(data=model_df,x='claim_status',hue='verified_status',ax=axes[0])
sns.countplot(data=model_df,x='claim_status',hue='author_ban_status',ax=axes[1])
plt.tight_layout();plt.show()

## 6. Statistical Analysis
H0: verified and unverified accounts have the same mean view count. H1: they differ. Alpha = 0.05.

In [87]:
a=model_df.loc[model_df['verified_status']=='verified','video_view_count']
b=model_df.loc[model_df['verified_status']=='not verified','video_view_count']
t,p=stats.ttest_ind(a,b,equal_var=False)
print(f't={t:.3f}; p={p:.4g}; decision:', 'Reject H0' if p<.05 else 'Fail to reject H0')

t=-25.499; p=2.609e-120; decision: Reject H0


## 7. Feature Engineering
Safe division avoids zero-denominator errors. Rates normalize engagement by view count.

### Likes per view

In [90]:
model_df['likes_per_view']=np.divide(model_df['video_like_count'],model_df['video_view_count'],out=np.zeros(len(model_df)),where=model_df['video_view_count'].to_numpy()!=0)

### Comments per view

In [92]:
model_df['comments_per_view']=np.divide(model_df['video_comment_count'],model_df['video_view_count'],out=np.zeros(len(model_df)),where=model_df['video_view_count'].to_numpy()!=0)

### Shares per view

In [94]:
model_df['shares_per_view']=np.divide(model_df['video_share_count'],model_df['video_view_count'],out=np.zeros(len(model_df)),where=model_df['video_view_count'].to_numpy()!=0)

### Text length

In [96]:
model_df['text_length']=model_df['video_transcription_text'].str.len()

### Duration band

In [98]:
model_df['duration_band']=pd.cut(model_df['video_duration_sec'],[0,15,30,45,60],labels=['0-15','16-30','31-45','46-60'],include_lowest=True)

### Engineered feature preview

In [100]:
model_df[['likes_per_view','comments_per_view','shares_per_view','text_length','duration_band']].head()

## 8. Training and Test Preparation
Positive class: claim equals 1. Pipelines ensure transformations are fitted only on training data.

### Define X

In [103]:
cols=['video_duration_sec','video_view_count','video_like_count','video_share_count','video_download_count','video_comment_count','likes_per_view','comments_per_view','shares_per_view','text_length','verified_status','author_ban_status','duration_band','video_transcription_text']
X=model_df[cols].copy();print(X.shape)

(19084, 14)


### Define y

In [105]:
y=(model_df['claim_status']=='claim').astype(int);print(y.value_counts().sort_index())

claim_status
0    9476
1    9608
Name: count, dtype: int64


### Stratified 80:20 split

In [107]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=RANDOM_STATE)

### Split dimensions

In [109]:
print(X_train.shape,X_test.shape,y_train.shape,y_test.shape)

(15267, 14) (3817, 14) (15267,) (3817,)


### Split target balance

In [111]:
balance=pd.DataFrame({'train':y_train.value_counts(normalize=True).sort_index()*100,'test':y_test.value_counts(normalize=True).sort_index()*100})
balance.index=balance.index.map({0:'opinion',1:'claim'});balance.round(2)

### Configure numerical, categorical, and text transformations

In [113]:
num=['video_duration_sec','video_view_count','video_like_count','video_share_count','video_download_count','video_comment_count','likes_per_view','comments_per_view','shares_per_view','text_length']
cat=['verified_status','author_ban_status','duration_band']
numeric_lr=Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
categorical=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))])
text=TfidfVectorizer(stop_words='english',ngram_range=(1,2),max_features=300,min_df=2)
pre_lr=ColumnTransformer([('numeric',numeric_lr,num),('categorical',categorical,cat),('text',text,'video_transcription_text')])
pre_rf=ColumnTransformer([('numeric',SimpleImputer(strategy='median'),num),('categorical',categorical,cat),('text',text,'video_transcription_text')])

## 9. Baseline Model
A DummyClassifier predicts only the most frequent class.

In [115]:
baseline=DummyClassifier(strategy='most_frequent').fit(X_train[['video_duration_sec']],y_train)

In [116]:
base_pred=baseline.predict(X_test[['video_duration_sec']])

In [117]:
base_acc=accuracy_score(y_test,base_pred);base_pre=precision_score(y_test,base_pred,zero_division=0);base_rec=recall_score(y_test,base_pred,zero_division=0);base_f1=f1_score(y_test,base_pred,zero_division=0)
print(f'Accuracy={base_acc:.4f}; Precision={base_pre:.4f}; Recall={base_rec:.4f}; F1={base_f1:.4f}')

Accuracy=0.5035; Precision=0.5035; Recall=1.0000; F1=0.6698


## 10. Logistic Regression

In [119]:
lr=Pipeline([('preprocessor',pre_lr),('model',LogisticRegression(max_iter=2000,class_weight='balanced',random_state=RANDOM_STATE))])

In [120]:
lr.fit(X_train,y_train)

In [121]:
lr_pred=lr.predict(X_test)

In [122]:
lr_prob=lr.predict_proba(X_test)[:,1]

### Logistic Regression Accuracy

In [124]:
lr_acc=accuracy_score(y_test,lr_pred);print(lr_acc)

0.9994760282944721


### Logistic Regression Precision

In [126]:
lr_pre=precision_score(y_test,lr_pred);print(lr_pre)

1.0


### Logistic Regression Recall

In [128]:
lr_rec=recall_score(y_test,lr_pred);print(lr_rec)

0.9989594172736732


### Logistic Regression F1-score

In [130]:
lr_f1=f1_score(y_test,lr_pred);print(lr_f1)

0.9994794377928162


### Logistic Regression Classification report

In [132]:
print(classification_report(y_test,lr_pred,target_names=['opinion','claim']))

              precision    recall  f1-score   support

     opinion       1.00      1.00      1.00      1895
       claim       1.00      1.00      1.00      1922

    accuracy                           1.00      3817
   macro avg       1.00      1.00      1.00      3817
weighted avg       1.00      1.00      1.00      3817



### Logistic Regression Confusion matrix

In [134]:
ConfusionMatrixDisplay.from_predictions(y_test,lr_pred,display_labels=['opinion','claim'],cmap='Blues');plt.title('Logistic Regression');plt.show()

### Logistic Regression ROC-AUC

In [136]:
lr_auc=roc_auc_score(y_test,lr_prob);print(lr_auc)

1.0


### Logistic Regression ROC curve

In [138]:
fpr_lr,tpr_lr,_=roc_curve(y_test,lr_prob)
plt.plot(fpr_lr,tpr_lr,label=f'AUC={lr_auc:.3f}');plt.plot([0,1],[0,1],'--');plt.legend();plt.title('Logistic Regression ROC');plt.show()

## 11. Random Forest Classifier

In [140]:
rf=Pipeline([('preprocessor',pre_rf),('model',RandomForestClassifier(n_estimators=200,min_samples_leaf=2,class_weight='balanced',random_state=RANDOM_STATE,n_jobs=-1))])

In [141]:
rf.fit(X_train,y_train)

In [142]:
rf_pred=rf.predict(X_test)

In [143]:
rf_prob=rf.predict_proba(X_test)[:,1]

### Random Forest Accuracy

In [145]:
rf_acc=accuracy_score(y_test,rf_pred);print(rf_acc)

1.0


### Random Forest Precision

In [147]:
rf_pre=precision_score(y_test,rf_pred);print(rf_pre)

1.0


### Random Forest Recall

In [149]:
rf_rec=recall_score(y_test,rf_pred);print(rf_rec)

1.0


### Random Forest F1-score

In [151]:
rf_f1=f1_score(y_test,rf_pred);print(rf_f1)

1.0


### Random Forest Classification report

In [153]:
print(classification_report(y_test,rf_pred,target_names=['opinion','claim']))

              precision    recall  f1-score   support

     opinion       1.00      1.00      1.00      1895
       claim       1.00      1.00      1.00      1922

    accuracy                           1.00      3817
   macro avg       1.00      1.00      1.00      3817
weighted avg       1.00      1.00      1.00      3817



### Random Forest Confusion matrix

In [155]:
ConfusionMatrixDisplay.from_predictions(y_test,rf_pred,display_labels=['opinion','claim'],cmap='Greens');plt.title('Random Forest');plt.show()

### Random Forest ROC-AUC

In [157]:
rf_auc=roc_auc_score(y_test,rf_prob);print(rf_auc)

1.0


### Random Forest feature importance

In [159]:
names=rf.named_steps['preprocessor'].get_feature_names_out()
importance=pd.Series(rf.named_steps['model'].feature_importances_,index=names).sort_values(ascending=False)
display(importance.head(15).to_frame('importance'))
importance.head(15).sort_values().plot.barh(figsize=(8,5));plt.title('Top 15 Feature Importances');plt.show()

## 12. Hyperparameter Tuning
RandomizedSearchCV uses training data only and chooses by claim F1-score.

In [161]:
params={'model__n_estimators':[150,250],'model__max_depth':[None,12,20],'model__min_samples_split':[2,5],'model__min_samples_leaf':[1,2],'model__max_features':['sqrt',.7]}

In [162]:
search=RandomizedSearchCV(Pipeline([('preprocessor',pre_rf),('model',RandomForestClassifier(class_weight='balanced',random_state=RANDOM_STATE,n_jobs=-1))]),params,n_iter=4,scoring='f1',cv=3,random_state=RANDOM_STATE,n_jobs=-1,refit=True)

In [163]:
search.fit(X_train,y_train)

In [164]:
print('Best CV F1:',search.best_score_);print(search.best_params_)

Best CV F1: 0.9992838923061903
{'model__n_estimators': 250, 'model__min_samples_split': 5, 'model__min_samples_leaf': 1, 'model__max_features': 0.7, 'model__max_depth': 12}


In [165]:
tuned=search.best_estimator_;t_pred=tuned.predict(X_test);t_prob=tuned.predict_proba(X_test)[:,1]
t_acc=accuracy_score(y_test,t_pred);t_pre=precision_score(y_test,t_pred);t_rec=recall_score(y_test,t_pred);t_f1=f1_score(y_test,t_pred);t_auc=roc_auc_score(y_test,t_prob)
print(f'Accuracy={t_acc:.4f}; Precision={t_pre:.4f}; Recall={t_rec:.4f}; F1={t_f1:.4f}; ROC-AUC={t_auc:.4f}')

Accuracy=1.0000; Precision=1.0000; Recall=1.0000; F1=1.0000; ROC-AUC=1.0000


## 13. Model Comparison

In [167]:
comparison=pd.DataFrame([
['Baseline',base_acc,base_pre,base_rec,base_f1,np.nan],
['Logistic Regression',lr_acc,lr_pre,lr_rec,lr_f1,lr_auc],
['Random Forest',rf_acc,rf_pre,rf_rec,rf_f1,rf_auc],
['Tuned Random Forest',t_acc,t_pre,t_rec,t_f1,t_auc]],columns=['Model','Accuracy','Precision','Recall','F1-score','ROC-AUC'])
comparison.set_index('Model').round(4)

### ROC comparison

In [169]:
fpr_rf,tpr_rf,_=roc_curve(y_test,rf_prob);fpr_t,tpr_t,_=roc_curve(y_test,t_prob)
plt.plot(fpr_lr,tpr_lr,label=f'Logistic {lr_auc:.3f}');plt.plot(fpr_rf,tpr_rf,label=f'RF {rf_auc:.3f}');plt.plot(fpr_t,tpr_t,label=f'Tuned RF {t_auc:.3f}')
plt.plot([0,1],[0,1],'--');plt.legend();plt.title('ROC Comparison');plt.show()

### Training versus test F1

In [171]:
train_f1=f1_score(y_train,tuned.predict(X_train))
pd.DataFrame({'Set':['Train','Test'],'Claim F1':[train_f1,t_f1]}).round(4)

## 14. Final Selection and Business Recommendations

False positive: an opinion is unnecessarily prioritised as a claim. False negative: a claim is deprioritised as an opinion. For safety-oriented moderation, recall for claims may deserve more weight than accuracy.


In [173]:
best=comparison.sort_values(['F1-score','Recall','ROC-AUC'],ascending=False).iloc[0]
views=model_df.groupby('claim_status')['video_view_count'].median()
print(f"Selected candidate: {best['Model']}")
print(f"Test F1={best['F1-score']:.4f}; recall={best['Recall']:.4f}; precision={best['Precision']:.4f}; AUC={best['ROC-AUC']:.4f}")
print(f"Median views: claim={views['claim']:,.0f}; opinion={views['opinion']:,.0f}")
print('Top RF feature:',importance.index[0])

Selected candidate: Random Forest
Test F1=1.0000; recall=1.0000; precision=1.0000; AUC=1.0000
Median views: claim=501,555; opinion=4,953
Top RF feature: numeric__video_view_count


### Portfolio Conclusion

This project used the supplied TikTok dataset to build a reproducible claim-versus-opinion classification workflow: data quality checks, EDA, statistical testing, feature engineering, train-only preprocessing pipelines, baseline comparison, Logistic Regression, Random Forest, tuning, and held-out evaluation.

Limitations: the dataset is synthetic; engagement values may be post-publication signals and can cause temporal leakage for real-time moderation. The model does not verify truth or determine policy violation. Use predictions to prioritize human review, then monitor drift, subgroup errors, and operational impact.
